# Lab Work - 6.4

**Dataset (8 samples, binary classification)**
- Samples: S1 to S8
- y_true = [1, 0, 1, 0, 1, 0, 1, 0]  
- y_proba = [0.95, 0.85, 0.78, 0.62, 0.55, 0.48, 0.30, 0.10]
- Positives (1): S1, S3, S5, S7
- Negatives (0): S2, S4, S6, S8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, average_precision_score

# Data
samples = ['S1','S2','S3','S4','S5','S6','S7','S8']
y_true = np.array([1, 0, 1, 0, 1, 0, 1, 0])
y_proba = np.array([0.95, 0.85, 0.78, 0.62, 0.55, 0.48, 0.30, 0.10])

print('Dataset loaded successfully!')
print('Number of positives:', y_true.sum())

## Q1: The ROC Curve

**01 Axes Definition**
- **x-axis (FPR)** = FP / (FP + TN) — Fraction of actual negatives incorrectly classified as positive.
- **y-axis (TPR)** = TP / (TP + FN) — Fraction of actual positives correctly classified as positive.

In [ ]:
# 03 Sorted list by descending probability
df = pd.DataFrame({'Sample': samples, 'y_true': y_true, 'y_proba': y_proba})
df_sorted = df.sort_values('y_proba', ascending=False).reset_index(drop=True)
print(df_sorted)

In [ ]:
# 04-06 Compute ROC points at every threshold
thresholds = np.concatenate(([1.0], np.sort(np.unique(y_proba))[::-1], [0.0]))

fprs, tprs, thresh_vals = [], [], []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fprs.append(fpr)
    tprs.append(tpr)
    thresh_vals.append(t)

roc_df = pd.DataFrame({'Threshold': thresh_vals, 'FPR': fprs, 'TPR': tprs})
print(roc_df)

In [ ]:
# Plot the step ROC curve
plt.figure(figsize=(10, 8))
plt.step(fprs, tprs, where='post', color='blue', label='ROC Curve (step)')
plt.plot([0, 1], [0, 1], 'r--', label='Random Chance (AUC=0.5)')

for i, t in enumerate(thresh_vals):
    plt.annotate(f'{t:.2f}', (fprs[i], tprs[i]), xytext=(5,5), textcoords='offset points')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve with Threshold Annotations')
plt.legend()
plt.grid(True)
plt.show()

## Q2: Computing AUC

In [ ]:
# Trapezoidal Rule (manual)
auc_trap = 0.0
for i in range(len(fprs)-1):
    width = fprs[i+1] - fprs[i]
    height = (tprs[i] + tprs[i+1]) / 2
    auc_trap += width * height

print('Manual Trapezoidal AUC:', round(auc_trap, 4))
print('sklearn AUC:', round(roc_auc_score(y_true, y_proba), 4))

In [ ]:
# Mann-Whitney / Wilcoxon interpretation
pos = y_proba[y_true == 1]
neg = y_proba[y_true == 0]
concordant = sum(p > n for p in pos for n in neg)
auc_mw = concordant / (len(pos) * len(neg))
print('Mann-Whitney AUC:', round(auc_mw, 4))

## Q3: AUC Interpretation

**Answers (fill in your own explanations as needed):**
1. AUC = P(positive score > negative score)
2. Random classifier → diagonal line
4. Perfect classifier → AUC = 1.0
6. AUC < 0.5 → worse than random (consider inverting predictions)
6. Threshold-invariant: good for comparing models across operating points
6. Use Precision-Recall AUC when positive class is rare or cost of FP/FN is asymmetric.

In [ ]:
# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_true, y_proba)
plt.figure(figsize=(8,6))
plt.step(rec, prec, where='post', color='green')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()
print('Average Precision (PR-AUC):', round(average_precision_score(y_true, y_proba), 4))

## Q4: Deep Intuition

**01** Classifier A is genuinely better because AUC=0.92 shows it ranks positives above negatives more reliably, while Accuracy=0.91 for B is misleading on the 90% negative dataset. Accuracy can look high on imbalanced data even when the model fails to detect the minority positive class.

**02** To catch at least 80% of fraud, raise the classification threshold until Recall ≥ 0.80 on the ROC-like score ranking; this will likely increase False Positives. The trade-off is lower Precision, meaning more legitimate transactions are flagged as fraud while fewer frauds are missed.

**03** AUC-ROC plots TPR versus FPR and measures ranking quality across thresholds; AUC-PR plots Precision versus Recall and focuses on positive-class performance when positives are rare. Use AUC-ROC for balanced data or when both classes matter; use AUC-PR for rare-event detection like fraud or disease screening.

**04** Report a single metric like AUC because it summarizes overall ranking performance without choosing a threshold. For balanced data with equal FP and FN cost, AUC is a threshold-independent measure of classifier quality and is easy for a non-technical stakeholder to compare.